In [ ]:
import ast
from glob import glob 

df = [] 

for file in glob('/workspace/VLMEval/results/spu_*.csv'): 
    
    tdf = pd.read_csv(file)
    tdf['model'] = file.split('/')[-1].split('.csv')[0].split('_')[1]
    df.append(tdf) 

df = pd.concat(df, axis=0) 
df['correct'] = df['output'].str.strip().str.lower().str[0] == df['answer'].str.strip().str.lower()
# if blind is in the model, replace -blind with '' and add column for blind =True 
df['blind'] = df['model'].str.contains('-blind') 
df['model'] = df['model'].str.replace('-blind', '') 
df['spurious_correlation_type'] = df['spurious_correlation_type'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df= df.explode('spurious_correlation_type') 

# Step 1: Count number of occurrences per type+model
count_df = df.groupby(['spurious_correlation_type', 'model'])['correct'].count().reset_index(name='count')

# Step 2: Filter for counts > 10
valid_pairs = count_df[count_df['count'] > 10][['spurious_correlation_type', 'model']]

# Step 3: Merge to filter original DataFrame
filtered_df = df.merge(valid_pairs, on=['spurious_correlation_type', 'model'], how='inner')

df  
df.groupby(['model'])['correct'].mean()
df.groupby(['spurious_correlation_type', 'model', 'blind'])['correct'] \
             .count() \
             .reset_index(name='count') \
             .query('count < 10').spurious_correlation_type.unique()